In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from loguru import logger
from typing import Tuple, List, Dict, Set, Any, Union, Optional, Annotated, Callable, Literal, TypeVar

logger.remove()
logger.add(sys.stdout)

In [ ]:
config_file_path: str = "/home/user/data-da-ds-de/data_validation_pandas/data/configs/master_data_setup config.xlsx"

sheet_names = pd.ExcelFile(config_file_path).sheet_names
sheet_names = [sheet_name for sheet_name in sheet_names if sheet_name != "Note"]
logger.info(sheet_names)

In [ ]:
df: pd.DataFrame = pd.read_excel(config_file_path, sheet_name="InternalSuspendedList")
df.head()

In [ ]:
# filter column need check
df = df[df["is_need_check"] == True]
df.head()

In [ ]:
# check is_preprocessing
df_is_preprocessing = df[df["is_preprocessing"] == True]
df_is_preprocessing.head()

# if need preprocessing:
# check separators
df_separators = df_is_preprocessing[df_is_preprocessing["seperators"].notna()]
df_separators

# check mapping
df_mapping = df_is_preprocessing[df_is_preprocessing["mapping"].notna()]
df_mapping

# check default value
df_default_value = df_is_preprocessing[df_is_preprocessing["default_value"].notna()]
df_default_value

In [ ]:
# mapping type
data_type_mapping: Dict = {
    "str": str,
    "int": int,
    "float": float,
    "bool": bool,
    "date": pd.to_datetime,
    "datetime": pd.to_datetime,
    "time": pd.to_datetime,
    "timedelta": pd.to_timedelta,
    "category": pd.Categorical,
    "object": str,
}
# if not in data_type_mapping: default as str

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# range check
df_range_check = df.loc[df["range_check"].notna(), ["type_check","range_check"]]
df_range_check["range_check"] = df_range_check["range_check"].astype(str).str.split("-").map(lambda x: [i.strip() for i in x])

# map to type_check -> transform str to correct data type

df_range_check["type_check"] = df_range_check["type_check"].map(data_type_mapping)
df_range_check["range_check"] = df_range_check.apply(
    lambda x: [
        x["type_check"](i) for i in x["range_check"]
        ],
        axis=1
    )

df_range_check

In [ ]:
df_range_check.loc[4, "range_check"][0]

In [ ]:
# value list check
df_value_list_check = df.loc[df["value_list_check"].notna(), ["type_check", "value_list_check"]]
df_value_list_check["value_list_check"] = df_value_list_check["value_list_check"].astype(str).str.split("\n").map(lambda x: [i.strip() for i in x])
df_value_list_check["type_check"] = df_value_list_check["type_check"].map(data_type_mapping)
df_value_list_check["value_list_check"] = df_value_list_check.apply(
    lambda x: [
        x["type_check"](i) for i in x["value_list_check"]
        ],
        axis=1
    )

df_value_list_check

In [ ]:
df_condition["condition"]

In [ ]:
# condition
def process_condition_string(string: str) -> list:
    list_condition = string.split("\n")
    list_condition = [i.strip() for i in list_condition]
    for i in range(len(list_condition)):
        if list_condition[i] in ["and", "or" "xor", "not"]:
            continue
        list_condition[i] = list_condition[i].replace("; ", "").replace("(", "").replace(")", "")
    query = "".join(list_condition)
    return query
df_condition = df.loc[df["condition"].notna(), ["condition"]]
df_condition["condition"] = df_condition["condition"].map(lambda x: process_condition_string(x))
df_condition

In [ ]:
# reference
file_path: str = "file_path"
def process_reference_tuple(string: str, default_filepath: str = file_path) -> list:
    list_condition = string.replace("(", "").replace(")", "").split("; ")
    list_condition = [i.strip() for i in list_condition]
    file_path = default_filepath
    if list_condition[0] != "*":
        file_path = list_condition[0]
    sheet_name = list_condition[1]
    column_name = list_condition[2]
    value_list = []
    if list_condition[-1] != "_":
    #     data = pd.read_excel(file_path, sheet_name=sheet_name) # replace by polars
    #     value_list = data[column_name].unique().tolist()
    # else:
        value_list = list_condition[-1].replace("[", "").replace("]", "").split(",")
        value_list = [i.strip() for i in value_list]
    return file_path, sheet_name, column_name, value_list



def process_reference_string(string: str) -> list:
    list_condition = string.split("\n")
    list_condition = [i.strip() for i in list_condition]
    for i in range(len(list_condition)):
        if list_condition[i] in ["and", "or" "xor", "not"]:
            continue
        list_condition[i] = process_reference_tuple(list_condition[i])

    return list_condition

df_reference = df.loc[df["is_refered"].notna(), ["reference_list"]]
df_reference
# target: list_condition[:-1]: file_path, sheet_name, column_name
# target: list_condition[-1]: value_list
# df_refered_1 = pd.read_excel(file_path, sheet_name=sheet_name)
# res_1 = df_refered_1[column_name].unique().tolist()
# df_refered_2 = pd.read_excel(file_path, sheet_name=sheet_name)
# res_2 = df_refered_2[column_name].unique().tolist()
# df.query(f"`{column_name}` in {res_1} and `{column_name}` in {res_2}")